In [ ]:
import os
import sys

IS_COLAB = "google.colab" in sys.modules
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if IS_COLAB:
    print("Colab environment for HW3 part 1.")
    from google.colab import drive
    drive.mount("/content/drive")
    HW_DIR = "/content/drive/MyDrive/CS189/hw/hw3-pt1"
    os.chdir(HW_DIR)
    %pip install -r colab_requirements.txt
elif IS_DATABRICKS:
    %pip install -r databricks_requirements.txt
else:
    print("Local development. Make sure your kernel has everything in requirements.txt.")


In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("autograd_and_optimizers_pt1.ipynb")

<h1>Homework 3 Part 1: Autograd and Optimizers</h1>

Welcome to Homework 3! In this assignment, you will use your newfound skills in backpropagation to implement an autodifferentiation library from scratch. You'll extend the single-variable autograd that we learned in lecture to function with general tensors, mimicking how `torch`'s autograd is implemented in research and industry.

---

## Due Date: Friday, October 23, 11:59 PM

This assignment is due on **Friday, October 23, 11:59 PM**. You must submit your work to the programming assignment by this deadline. Please refer to the syllabus for the [Slip Day policy](https://eecs189.org/fa26/syllabus/#slip-days). No late submissions will be accepted beyond the details outlined in the Slip Day policy.

### Submission Tips:
- **Plan ahead**: We strongly encourage you to submit your work several hours before the deadline. This will give you ample time to address any submission issues.
- **Reach out for help early**: If you encounter difficulties, contact course staff well before the deadline. While we are happy to assist with submission issues, we cannot guarantee responses to last-minute requests.

---

### Key Learning Objectives:
1. Learn how backpropagation is implemented in libraries like PyTorch
2. Understand how optimizers are implemented, and the advantages of Adam 
3. Use your custom autograd and backprop implementations to train a model

---

<div align="center">

### Grading Breakdown

| Question  | Manual Grading? | Points |
| --------- | --------------- | ------ |
| q1        | No              | 10     |
| q2a       | No              | 10     |
| q2b       | No              | 10     |
| q3a       | No              | 3      |
| q3b       | No              | 3      |
| q3c       | No              | 5      |
| q4 (plot) | Yes             | 2      |
| q4        | No              | 10      |
| **Total** |                 | **53** |

</div>

---

### Instructions:
1. Carefully read each question and its requirements.
2. Complete all TODOs in the notebook. You may add extra lines of code if needed.
3. For manual questions, write your answer in the provided answer cell or save the requested figure; it is saved to `written/` and included in your submission.
4. Test your code thoroughly to ensure it meets the requirements.


---

### Autograding
- After each coding question, run `grader.check("q...")` to execute the **public** tests.
- Keep the `tests/` folder next to this notebook. It ships in the homework repo. If staff announces a test update, use the cell below to restore only `tests/` from the announced tag.
- The autograder also runs additional tests that are not in this handout. You will see pass/fail and error messages for those tests when you submit.
- Submit your coding `.ipynb` and output `.png` and `.txt` files to the **programming** assignment. The export cell at the end of the notebook packages everything into a `.zip` you can submit and warns you of missing files.
- Both the final export cell and the autograder expect the assignment notebook to have the same name as the input to `otter.Notebook(...)` in the Otter initialization cell at the top of the notebook. Do not rename your notebook, otherwise the final export and the autograder may run into errors processing your submission!
- A PDF of your manually graded responses is uploaded to the **associated paper assignment**. You do not need to select pages.

Good luck!


## Student instructions for updating tests

- Keep the `tests/` folder next to this notebook these are part of the homework repo.
- Run public checks with `grader.check("q1")` or `grader.check("q3a")`.
- If staff announces a public-test update, set `TESTS_VERSION` to the announced tag, set `REPULL_TESTS = True` in the cell below, run it, then set it back to `False`. That restores only `tests/` from the tag and leaves your notebook and other files alone.
- Submit the exported programming zip. Written responses and figures are collected from `written/`.

In [ ]:
# PUBLIC TEST VERSION
TESTS_VERSION = "hw3-public-v1"
REPULL_TESTS = False

if REPULL_TESTS:
    !git fetch origin tag {TESTS_VERSION}
    !git restore --source={TESTS_VERSION} -- tests/


### **IMPORTANT:**
- Do not change the random seed values.
- Save the notebook before exporting.
- You do not have to save or submit any trained models for this homework assignment.

In [ ]:
from __future__ import annotations
from collections import deque
from dataclasses import dataclass
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
try:
    import typst
except ImportError:
    typst = None

pd.options.plotting.backend = "plotly"

# Set random seeds for reproducible results
SEED = 189
np.random.seed(SEED)
IS_GRADING_ENV = os.getenv("IS_GRADING_ENV") == "true"

WRITTEN_DIR = "written"
os.makedirs(WRITTEN_DIR, exist_ok=True)

written_files_path = os.path.join("tests", "written_files.json")
if os.path.exists(written_files_path):
    with open(written_files_path, encoding="utf-8") as f:
        EXPECTED_WRITTEN_FILES = json.load(f)
else:
    EXPECTED_WRITTEN_FILES = []

def save_figure(fig, question_name):
    """Save a Plotly or Matplotlib figure under written/."""
    if IS_GRADING_ENV:
        return None
    path = os.path.join(WRITTEN_DIR, f"{question_name}.png")
    try:
        if hasattr(fig, "write_image"):
            fig.write_image(path)
        else:
            fig.savefig(path, bbox_inches="tight")
    except Exception as e:
        print(f"WARNING: Could not save {path}: {e}")
        return None
    print(f"Saved {path}")
    return path

def save_written_answer(answer, question_name):
    """Save a written response under written/."""
    if IS_GRADING_ENV:
        return None
    path = os.path.join(WRITTEN_DIR, f"{question_name}.txt")
    try:
        with open(path, "w", encoding="utf-8") as f:
            f.write(str(answer).strip() + "\n")
    except Exception as e:
        print(f"WARNING: Could not save {path}: {e}")
        return None
    print(f"Saved {path}")
    return path

def warn_missing_written_files():
    """Report expected written artifacts that are missing locally."""
    missing = [name for name in EXPECTED_WRITTEN_FILES if not os.path.isfile(os.path.join(WRITTEN_DIR, name))]
    if missing:
        print("WARNING: Missing expected files in written/:")
        for name in missing:
            print(f"  - {name}")
        print("Re-run the cells that save these artifacts before exporting.")
    else:
        print(f"All {len(EXPECTED_WRITTEN_FILES)} expected files found in written/.")
    return missing


# Question 1: Building the Computation Graph

Recall that a computation graph is a directed graph that represents how data flows through a mathematical calculation. Under the hood, machine learning models are just doing a bunch of math on their input data to produce some final output!

Our model needs some way to build a computation graph for backpropagation so that it knows where to pass down gradients during the backwards pass. The way that PyTorch does this is that every time you combine two values to compute a third one in the forward pass, it will keep track of the operation internally. In this homework assignment, we will be using `numpy` and implementing our own version of PyTorch autograd.

**Note:** You might hear about both "autodiff" and "autograd". What's the difference? Autodiff refers to the general concept of "automatically" evaluating the derivative of a function written in code. By automatically, we mean systematically applying the chain rule rather than using finite numerical estimates (e.g. nudging the input values by a tiny amount, and measuring how the output's numerical values change). Autograd was originally the name of a specific Python package for NumPy-based differentiation, but now it's often used to refer to *any* kind of software that can perform autodifferentiation, such as the autodiff engine built into the PyTorch framework.

## Introducing BearTensor

Before we can dig into our multivariable calculus, we first need a good abstraction for tensors,
which are just $n$-dimensional arrays. `numpy` doesn't provide autodifferentiation for its `ndarray`, so we can build a wrapper around it. For this homework we are going to be using the `BearTensor` class, which you can think of as a custom implementation of `torch.Tensor` that acts as a wrapper around `numpy` arrays.

Just like how PyTorch keeps track of every mathematical operation that is performed on a `torch.tensor` so that it can efficiently build a computation graph, we want to implement a similar functionality for `BearTensor`; everytime we add, subtract, divide, etc. two `BearTensor` to get a third one, we should keep track of the fact that the third tensor's **parents** are the two source tensors. `BearParent` and `BearGrad` are additional classes representing the operations and relationships among nodes in a computation graph that we've already defined for you.

Methods like `__add__` and `__matmul__` are Python's special ("dunder") methods: Python calls them when it sees the matching operator. With `BearTensor` inputs, `x + y` calls `x.__add__(y)`, `x @ y` calls `x.__matmul__(y)`, and `x ** 2` calls `x.__pow__(2)`. In `x * y + x`, Python first calls `x.__mul__(y)`, then calls `__add__(x)` on that result. This means that we can use `BearTensors` with everyday math expressions, and the methods (once you implement them :)) will compute the resulting value and record the gradients needed to build the compuation graph behind the scenes!

You create a tensor with `BearTensor(value, name)`. The `name` is optional and is only used to label nodes when printing or drawing the graph. Each tensor also has an **adjoint**: the gradient $\partial L / \partial x$ of the final output $L$ with respect to that tensor $x$, stored in `x.adjoint` with the same shape as `x.value`. PyTorch stores the same quantity in a tensor's `.grad`.

Here is an example of what a computation graph might look like from lecture:
<div style="text-align: center;">
  <img src="https://imgur.com/PO3S6TI.png" alt="Backpropagation" style="display: block; margin-left: auto; margin-right: auto; width: 60%;">
</div>

---

Currently, our `BearTensor` has no operations. Implement the following functions in the class:
- Addition (`__add__`)
- Subtraction (`__sub__`)
- Multiplication (`__mul__`)
- Power (`__pow__`)
- Matrix multiplication (`__matmul__`)
  - **Hint:** refer to Section 2.4 of [The Matrix Cookbook](https://www.math.uwaterloo.ca/~hwolkowi/matrixcookbook.pdf) for matrix/vector derivative formulas!
- Dot product (`dot`)
- Sum (`sum`)
- Mean (`mean`)
- ReLU (`relu`)
- Sigmoid (`sigmoid`)

---

A few things to remember:
1. You should return a new `BearTensor` from these functions without modifying the existing one
2. Set the new tensor's `parents` to its input tensor(s), with one `BearParent(input, BearGrad(fn, op_str))` per input; this is how we build the computation graph
3. Each `fn` takes the adjoint of the new tensor and returns that input's share of the gradient (the chain rule for one edge); this is how gradients flow during the backward pass

Pay close attention to array shapes in this homework:
- Every adjoint must have the same shape as its tensor's `value`.
- `(N,)` and `(N, 1)` are different shapes. But when they are mixed, NumPy will silently broadcast the result into `(N, N)`. To avoid bugs from silent broadcasting, we will implement `+`, `-` and `*` such that they require matching shapes!
- `sum`, `mean` and `dot` reduce to a single number. Return it as a 0-d array, for example `np.array(x.sum())` with shape `()`. This is what PyTorch does too.
- `@` follows NumPy's rules for 1-D operands: a 1-D array on the left acts as a row vector `(1, k)`, and a 1-D array on the right acts as a column vector `(k, 1)`. The provided code at the top of `__matmul__` reshapes these 1-D vector edge cases for you, so your gradient formulas only need to handle 2-D matrices. Remember to reshape each gradient back to the shape of its input.

**Note**: The public tests that we've provided in this notebook are NOT fully-comprehensive and often very simple. You should build more complex graphs and manually verify that your computed gradients and values are correct. Our hidden tests are more comprehensive and will check for full correctness 

In [ ]:
@dataclass
class BearGrad:
    '''How to send gradient back along one edge of the computation graph.
    - `fn` takes the adjoint of the output tensor and returns this parent's share of the gradient (the chain rule for one edge).
    - `op_str` is a label for the local derivative. It is only used when drawing the graph; you don't need it for this HW, but it can help with debugging.
    '''
    fn: callable
    op_str: str | None = None

@dataclass
class BearParent:
    '''One parent of a node: a tensor it was computed from, plus how to send gradient back to it.
    - `parent` is the parent `BearTensor`.
    - `grad` is the `BearGrad` for this edge; `grad.fn` computes the gradient passed back to `parent`.
    '''
    parent: BearTensor
    grad: BearGrad

    def __hash__(self):
        return id(self)

    def __eq__(self, other):
        return self is other

class BearTensor:
    '''A node in our computation graph.
    - `value` is the underlying data (a float NumPy array), computed during the forward pass.
    - `name` is an optional label, used only for printing and drawing the graph.
    - `parents` lists the tensors this one was computed from (empty for leaves, such as parameters and inputs).
    - `adjoint` is dL/d(this tensor): the gradient of the final output with respect to this tensor,
      with the same shape as `value`. It is computed in the backward pass. PyTorch calls this `.grad`.
    '''
    def __init__(self, value: np.ndarray, name: str | None = None, parents: list[BearParent] | None = None):
        """Wrap `value` as a float array. Leaves (parameters, inputs) have no parents."""
        self.value = np.asarray(value, dtype=float)
        self.name = name if name is not None else "tensor"
        self.parents = parents if parents is not None else []
        self.adjoint: float | np.ndarray = 0.0

    def __add__(self, other: BearTensor) -> BearTensor:
        """Elementwise self + other. Shapes must match."""
        if self.value.shape != other.value.shape:
            raise ValueError("Shapes must match")
        ...

    def __sub__(self, other: BearTensor) -> BearTensor:
        """Elementwise self - other. Shapes must match."""
        if self.value.shape != other.value.shape:
            raise ValueError("Shapes must match")
        ...

    def __mul__(self, other: BearTensor) -> BearTensor:
        """Elementwise self * other. Shapes must match."""
        if self.value.shape != other.value.shape:
            raise ValueError("Shapes must match")
        ...

    def __pow__(self, power: float) -> BearTensor:
        """Elementwise self ** power, for a constant power."""
        ...

    def __matmul__(self, other: BearTensor) -> BearTensor:
        """Matrix product self @ other. 1-D operands follow NumPy's rules."""
        # In NumPy, if the operand on the left is a 1-D vector, NumPy treats it as a row vector (1, k).
        # Similarly, if the operand on the right is a 1-D vector, NumPy treates it as a column vector (k, 1).
        # Then, NumPy removes the added dimension(s) from the result after matmul-ing the two vectors together.
        # Below, we reshape A and B into 2-D forms so that we can use the same generalized matrix gradient
        # formula when the inputs are vectors and when the inputs are matrices.
        # 
        # The forward product should still use the original inputs.
        #
        # Since NumPy's result and its adjoint may be scalar or 1-D,
        # `as_2d_adj` reshapes the adjoint to the 2-D output shape of A @ B.
        #
        # Make sure to reshape each computed gradient to its input's original shape before
        # returning it.

        A = self.value if self.value.ndim == 2 else self.value.reshape(1, -1)
        B = other.value if other.value.ndim == 2 else other.value.reshape(-1, 1)

        def as_2d_adj(adj):
            """View the adjoint of A @ B as a 2-D array of shape (rows of A, columns of B)."""
            return np.asarray(adj).reshape(A.shape[0], B.shape[1])

        ...

    def dot(self, other: BearTensor) -> BearTensor:
        """Inner product of two 1-D tensors. Returns a 0-d tensor."""
        if self.value.ndim != 1 or other.value.ndim != 1:
            raise ValueError("dot() only supports 1-D BearTensors (like torch.dot).")
        ...

    def sum(self) -> BearTensor:
        """Sum of all elements. Returns a 0-d tensor."""
        ...

    def mean(self) -> BearTensor:
        """Mean of all elements. Returns a 0-d tensor."""
        ...

    def relu(self) -> BearTensor:
        """Elementwise max(x, 0)."""
        ...

    def sigmoid(self) -> BearTensor:
        """Elementwise 1 / (1 + exp(-x))."""
        ...

**Disclaimer**: The Q1 test cases will NOT test that your adjoint calculations are correct until you implement Q2; if you are struggling with the next question, we recommend also looking back at your solution to Q1.

In [ ]:
grader.check("q1")

## Visualizing your computation graphs

Below, we have provided some code that will allow you to visualize your computation graphs and save a PDF of the visualized graph. There is an example of how you can use this helper code in the next cell! You are not required to use nor understand how this works, but it may help you debug while working on this homework.

In [ ]:
# Staff-provided helper!
# Run this cell, no code changes required :)

def to_typst_math(s: str) -> str:
    processed_s = s
    processed_s = processed_s.replace(" @ ", " ")
    processed_s = re.sub(r"([a-zA-Z])([0-9]+)", r"\1_\2", processed_s)
    processed_s = re.sub(r"\.([a-zA-Z]+)", r'."\1"', processed_s)
    processed_s = re.sub(r"([a-zA-Z]+)_([a-zA-Z_]+)", r'\1_"\2"', processed_s)
    processed_s = re.sub(r"\b(relu|sum)\b", r'"\1"', processed_s)
    return processed_s


def _traverse_graph(
    root: BearTensor,
) -> tuple[list[BearTensor], list[tuple[BearTensor, BearTensor, str]]]:
    nodes, edges = set(), set()
    visited = set()

    def build(v):
        if v not in visited:
            visited.add(v)
            nodes.add(v)
            for p in v.parents:
                edges.add((p.parent, v, p.grad.op_str))
                build(p.parent)

    build(root)
    return list(nodes), list(edges)


def _minimize_crossings(
    nodes_by_level: list[list[BearTensor]],
    edges: list[tuple[BearTensor, BearTensor, str]],
    iterations: int = 4,
) -> list[list[BearTensor]]:
    node_to_parents = {n: [] for level in nodes_by_level for n in level}
    node_to_children = {n: [] for level in nodes_by_level for n in level}
    for p, c, _ in edges:
        if p in node_to_children and c in node_to_parents:
            node_to_children[p].append(c)
            node_to_parents[c].append(p)

    ordered_levels = [list(level) for level in nodes_by_level]

    for _ in range(iterations):
        for level_idx in range(1, len(ordered_levels)):
            child_level = ordered_levels[level_idx - 1]
            child_ranks = {node: rank for rank, node in enumerate(child_level)}

            def barycenter_down(node):
                children = node_to_children.get(node, [])
                if not children:
                    return -1
                avg_rank = sum(child_ranks.get(c, 0) for c in children) / len(children)
                return avg_rank

            ordered_levels[level_idx].sort(key=barycenter_down)

        for level_idx in range(len(ordered_levels) - 2, -1, -1):
            parent_level = ordered_levels[level_idx + 1]
            parent_ranks = {node: rank for rank, node in enumerate(parent_level)}

            def barycenter_up(node):
                parents = node_to_parents.get(node, [])
                if not parents:
                    return -1
                avg_rank = sum(parent_ranks.get(p, 0) for p in parents) / len(parents)
                return avg_rank

            ordered_levels[level_idx].sort(key=barycenter_up)

    return ordered_levels


def _assign_node_levels(
    root: BearTensor,
    nodes: list[BearTensor],
    edges: list[tuple[BearTensor, BearTensor, str]],
) -> dict[BearTensor, tuple[float, float]]:
    node_to_id = {node: i for i, node in enumerate(nodes)}
    id_to_node = {i: node for i, node in enumerate(nodes)}

    adj = {node_to_id[n]: [] for n in nodes}
    for p, c, _ in edges:
        if c in node_to_id and p in node_to_id:
            adj[node_to_id[c]].append(node_to_id[p])

    levels = {n: -1 for n in nodes}
    if root in levels:
        levels[root] = 0

    q = deque([root])
    max_level = 0
    visited_bfs = {root}

    while q:
        u = q.popleft()
        u_id = node_to_id[u]
        for v_id in adj.get(u_id, []):
            v = id_to_node.get(v_id)
            if v and v not in visited_bfs:
                levels[v] = levels[u] + 1
                max_level = max(max_level, levels[v])
                visited_bfs.add(v)
                q.append(v)

    nodes_by_level = [[] for _ in range(max_level + 1)]
    for node, level in levels.items():
        if level != -1:
            nodes_by_level[level].append(node)

    for level_nodes in nodes_by_level:
        level_nodes.sort(key=lambda n: n.name)

    nodes_by_level = _minimize_crossings(nodes_by_level, edges)

    node_coords = {}
    for level, nodes_in_level in enumerate(nodes_by_level):
        num_in_level = len(nodes_in_level)
        for rank, node in enumerate(nodes_in_level):
            y_pos = rank - (num_in_level - 1) / 2.0
            x_pos = -level
            node_coords[node] = (x_pos, y_pos)

    return node_coords


def _generate_typst_source(
    nodes: list[BearTensor],
    edges: list[tuple[BearTensor, BearTensor, str]],
    coords: dict[BearTensor, tuple[float, float]],
) -> str:
    node_to_id = {node: i for i, node in enumerate(nodes)}

    header = """#import "@preview/fletcher:0.5.8": diagram, node, edge
#set page(width: auto, height: auto, margin: 10mm, fill: white)
#set text(font: "Linux Libertine", size: 10pt)

#diagram(cell-size: (40mm, 40mm), {
"""

    node_definitions = ""
    for n in nodes:
        if n not in coords:
            continue
        x, y = coords[n]
        node_id = node_to_id[n]
        math_name = to_typst_math(n.name)
        shape_str = str(n.value.shape)
        content = f"""block(stroke: 0.5pt, inset: 8pt, radius: 4pt, [\n#align(center, ${math_name}$)\n#line(length: 100%)\n#text(size: 9pt, `shape: {shape_str}`)\n])"""
        node_definitions += f'  node(({x}, {y}), {content}, name: "{node_id}")\n'

    edge_definitions = ""
    LABEL_LENGTH_THRESHOLD = 20

    for parent, child, op_str in edges:
        from_id = node_to_id.get(parent)
        to_id = node_to_id.get(child)
        if from_id is None or to_id is None:
            continue

        op_str = op_str or ""
        math_op_str = to_typst_math(op_str)
        label = f"${math_op_str}$"

        if len(op_str) > LABEL_LENGTH_THRESHOLD:
            edge_definitions += f'  edge(label("{from_id}"), label("{to_id}"), "->", label: {label}, label-sep: 4em)\n'
        else:
            edge_definitions += (
                f'  edge(label("{from_id}"), label("{to_id}"), "->", label: {label})\n'
            )

    footer = "})"

    return header + node_definitions + edge_definitions + footer


def draw_graph(root: BearTensor, output_typ_path: str = "computational_graph.typ"):
    nodes, edges = _traverse_graph(root)
    coords = _assign_node_levels(root, nodes, edges)
    typst_source = _generate_typst_source(nodes, edges, coords)

    with open(output_typ_path, "w", encoding="utf-8") as f:
        f.write(typst_source)
    output_pdf_path = output_typ_path.replace(".typ", ".pdf")
    typst.compile(output_typ_path, output=output_pdf_path)
    print(f"Saved PDF of graph to {output_pdf_path}")

In [ ]:
# Example code (feel free to play around and create your own BearTensor computation graphs)
a = BearTensor(np.array([2, 3]), "a")
b = BearTensor(np.array([1, 1]), "b")
c = a + b
draw_graph(c, "demo_graph.typ")

## Question 2: Implementing the Backward Pass

After solving the previous question, your `BearTensor` should be able to construct the computation graph from the forward pass. In this part, we will implement the backwards pass, using the `BearGrad` functionality from above to propagate the gradients downstream.

A computation graph records how each tensor is built from other tensors. During backpropagation, gradients start at the loss and are passed to the tensors that produced it. A tensor may be used by several later operations, so we must collect every contribution to its gradient before passing that gradient to its own parents.

Usually, a topological order puts each node before the nodes that depend on it. In this notebook, that means a tensor’s parents come before the tensor, so the loss would come last. You should implement `topological_sort(node)` in this standard order. Then, when you implement backpropagation in the `backward` function, you'll traverse the topological sort ordering in reverse (since ) backpropagation starts at the loss and moves toward each tensor's parents, AKA the opposite direction of topological sort).

## Question 2a: Topological Sort

Implement `topological_sort(node)`. Return every node reachable from `node` exactly once, ordered so that each node appears after its parents, with `node` last.

**Hint:** Topological sort can be implemented via a depth-first search that appends each node to a stack after visiting all of its parents!

In [ ]:
def topological_sort(node):
    '''Return all nodes reachable from `node` in topological order: parents before children, with `node` coming last.'''
    ...
    return sorted_nodes

In [ ]:
grader.check("q2a")

## Question 2b: Backpropagation

Training repeats the forward and backward pass many times. As in PyTorch, `backward()` **adds** into the adjoints of leaf tensors (tensors with no parents, such as parameters and inputs), but it doesn't overwrite them. This way, gradients can be summed over several minibatches. This is why training loops call `optimizer.zero_grad()` before each backward pass. You'll implement the `zero_grad()` method on the optimizer soon in Q3!

Intermediate nodes (tensors with one or more parents, such as tensors produced by an operation in the middle of our computation graph or the output tensor), use their adjoints to collect gradient contributions while the gradients are being passed through the graph during a single backward pass. `backward()` should reset these adjoints before starting, so values left over from an earlier pass are not propagated again.

Fill out `reset_intermediate_adjoints` and `backward`.

In [ ]:
def reset_intermediate_adjoints(self):
    """Reset the adjoints of non-leaf tensors in self's computation graph."""
    ...

def backward(self):
    """Accumulate gradients through self's computation graph."""
    # Hint: After resetting intermediate adjoints, what should self.adjoint be?
    # What is the derivative of self with respect to itself?
    ...

BearTensor.reset_intermediate_adjoints = reset_intermediate_adjoints
BearTensor.backward = backward

In [ ]:
grader.check("q2b")

# Question 3: Optimizers

Now that we are able to compute the gradients for all the parameters in our model, we want to be able to optimize them. In lecture we talked about Gradient Descent; in this homework, we will implement 3 widely-used optimizers, Stochastic Gradient Descent (SGD), SGD with Momentum, and Adam. It is highly recommended that you complete the paper/written portion of the homework before attempting this question.


## Question 3a: SGD

Remember that gradients point in the direction of fastest increase! But if we want to minimize a loss function, we want to go in the direction where it *decreases* the fastest. Therefore, we want to move in the *opposite* direction of the gradient.

Stochastic gradient descent does just that! It updates each parameter by moving it in the opposite of its gradient. This simple update rule is the foundation for the more adaptive optimizers in the next two subquestions.

A typical model training loop looks like this (we will go into more detail soon in the next notebook — stay tuned :)):

```python
optimizer.zero_grad()  # clear old gradients
loss = ...             # forward pass
loss.backward()        # compute the gradient for each parameter
optimizer.step()       # update parameter values
```

In this notebook:
- `p.value` is the parameter being learned.
- `p.adjoint` is its gradient.
- `self.lr` is the learning rate.
- `step()` updates parameters in place; it does not compute gradients or reset them.

The provided `Optimizer` parent class supplies shared setup and `zero_grad()`. `SGD`, `Momentum`, and `Adam` inherit from it, so you only need to implement each subclass's `step()` method:
- **SGD:** update each parameter by taking a step in the opposite direction of its gradient.
- **Momentum:** maintains a velocity for each parameter and uses it to smooth updates.
- **Adam:** tracks first- and second-moment estimates of the gradients and uses them for adaptive updates.

All three optimizers use the same `step()` interface but they differ in how they compute the update.

Implement the SGD optimizer.

In [ ]:
class Optimizer:
    def __init__(self, params: list[BearTensor], lr: float):
        self.params = params
        self.lr = lr

    def zero_grad(self):
        for p in self.params:
            p.adjoint = np.zeros_like(p.value)

    def step(self):
        raise NotImplementedError

class SGD(Optimizer):
    def __init__(self, params: list[BearTensor], lr: float):
        super().__init__(params, lr)

    def step(self):
        ...

In [ ]:
grader.check("q3a")

## Question 3b: Momentum

Momentum keeps a running update direction so that optimization can build speed along consistent slopes and reduce small oscillations. Implement the optimizer's stateful update and apply it to each parameter.

For each parameter $\theta$ with gradient $g$ (`p.adjoint`), momentum keeps a velocity $v$ (`self.velocities[i]`, starting at $0$) and updates

$$v_t = \beta\, v_{t-1} + g_t, \qquad \theta_t = \theta_{t-1} - \eta\, v_t,$$

where $\beta$ is `self.beta` and $\eta$ is `self.lr`. This is PyTorch's convention. Bishop writes an equivalent form with the learning rate folded into the velocity; with a constant $\eta$, both produce the same parameter updates.

Implement the Momentum optimizer.

In [ ]:
class Momentum(Optimizer):
    def __init__(self, params: list[BearTensor], lr: float, beta: float = 0.9):
        super().__init__(params, lr)
        self.beta = beta
        self.velocities = [np.zeros_like(p.value) for p in self.params]

    def step(self):
        ...

In [ ]:
grader.check("q3b")

## Question 3c: Adam

Adam combines momentum with per-parameter step-size adaptation by tracking running estimates of the first and second gradient moments. Implement these updates, including the time-dependent bias correction.

The slides call Adam's first- and second-moment estimates $s_i$ and $r_i$. In the code, these are stored as `self.s` and `self.r` (one array per parameter tensor), both initialized to $0$. Here $r_i$ is the running estimate of squared gradients, not the momentum velocity from Question 3b. For each parameter coordinate $i$ with gradient $g_i$ (`p.adjoint`), at step $t$ (`self.t`, counting from 1):

$$s_i^{(t)} = \beta_1 s_i^{(t-1)} + (1-\beta_1)\, g_i^{(t)}, \qquad r_i^{(t)} = \beta_2 r_i^{(t-1)} + (1-\beta_2)\, (g_i^{(t)})^2$$

$$\hat{s}_i^{(t)} = \frac{s_i^{(t)}}{1-\beta_1^t}, \qquad \hat{r}_i^{(t)} = \frac{r_i^{(t)}}{1-\beta_2^t}, \qquad \theta_i^{(t)} = \theta_i^{(t-1)} - \eta\, \frac{\hat{s}_i^{(t)}}{\sqrt{\hat{r}_i^{(t)}} + \epsilon}$$

The squares, square root and division are elementwise. Dividing by $1-\beta^t$ corrects for $s_i$ and $r_i$ starting at zero.

Implement the Adam optimizer.

In [ ]:
class Adam(Optimizer):
    def __init__(
        self,
        params: list[BearTensor],
        lr: float,
        beta1: float = 0.9,
        beta2: float = 0.999,
        eps: float = 1e-8,
    ):
        super().__init__(params, lr)
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.t = 0
        self.s = [np.zeros_like(p.value) for p in self.params]
        self.r = [np.zeros_like(p.value) for p in self.params]

    def step(self):
        ...

In [ ]:
grader.check("q3c")

## Optimization Exploration!

The code below compares the convergence rates of all 3 optimizers. Play around with the learning rates and other hyperparameters to see how the results change.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

def compare_optimizers(optimizers):
    np.random.seed(0)
    X = BearTensor(np.random.randn(1000, 24), "X")
    y = BearTensor(np.random.randn(1000, 1), "y")
    
    n_steps = 100
    results = {}

    W1_init = np.random.randn(24, 12)
    W2_init = np.random.randn(12, 1)

    for opt_class, title, lr in optimizers:
        W1 = BearTensor(deepcopy(W1_init), "W1")
        W2 = BearTensor(deepcopy(W2_init), "W2")
        params = [W1, W2]

        optimizer = opt_class(params, lr=lr)
        losses = []

        for step in range(n_steps):
            optimizer.zero_grad()

            hidden = (X @ W1).sigmoid()
            output = hidden @ W2
            loss = ((output - y) ** 2).mean()

            loss.backward()
            losses.append(loss.value.item())
            optimizer.step()

        results[title] = losses

    plt.figure(figsize=(8, 5))
    for title, losses in results.items():
        plt.plot(losses, label=title, linewidth=2)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Optimizer Convergence Comparison (Same Init)")
    plt.legend()
    plt.grid(True)
    plt.show()

compare_optimizers([
    (SGD, "SGD", 0.05),
    (Momentum, "Momentum", 0.05),
    (Adam, "Adam", 0.05)
])

# Question 4: Train a Model with your BearTensors!

In this section, we're going to use the backpropagation engine and optimizers you created to train a model to predict wine quality scores!

Training our `BearTensor` model is going to be slightly more involved than just using PyTorch. Our simple implementation does not allow us to define layers. Instead, we are going to define `BearTensors`, combine them manually to do a forward pass (which will automatically construct a computation graph!) and then finally backpropagate on our loss.

We are going to use the Wine Quality dataset, which contains 1599 samples of red wine. Each sample has 11 features representing various chemical properties, and the task is to predict the wine quality score (between 0 and 10). You can choose any loss function and optimizer (as long as you implemented it!) of your choice, feel free to play around with hyperparameters and different model sizes!

**Your Tasks:**

Train a model to predict wine quality scores. Your model should:
- Use at least one hidden layer
- Use an activation function for hidden layers
- Achieve a mean squared error (MSE) of ≤ 0.5 on the held-out test set (for reference, always predicting the average quality gives about 0.65)
- Save a plot of your train and test loss curves to `q4_loss_curves.png`

**You need to fill in the `predict()` function that uses your architecture to return a prediction given an input datapoint. The autograder will use this to test your implementation**.

**Note**: If your model takes more than 30 seconds to train it could crash the autograder; you should easily be able to **achieve a test MSE of ≤ 0.5** with a 1-hidden-layer neural net for full credit. The staff solution takes <1 second to train, so if your code is slow, you may be constructing your graph inefficiently.

**You will receive 50% credit for this question if your test MSE is ≤ 0.6.**

Tips for training models:
- Call `optimizer.zero_grad()` before each `backward()`; parameter adjoints accumulate across backward passes
- Quality scores are around 5–6, so if your output bias starts at 0, the early epochs are spent just learning the average. A larger learning rate, or initializing the output bias to `y_train.mean()`, can help
- Output your training and test loss every epoch; are they smoothly going down? If not, you may need to change the learning rate
- Ensure your computation graph is being built properly (remember our visualization helper functions from earlier!)
- Changing the number of layers or adding/removing bias terms can be a useful way to control how much your model is overfitting/underfitting

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# DO NOT CHANGE THE PREPROCESSING CODE
data = fetch_openml("wine-quality-red", as_frame=True)
X = data.data.to_numpy()
y = data.target.to_numpy().astype(float)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.4, random_state=42, shuffle=True
)

X_tensor = BearTensor(X_train, "X_train")               # shape: (N_train, input_dim)
y_tensor = BearTensor(y_train.reshape(-1,1), "y_train") # shape: (N_train,1)

# TODO: Add your training code here!
# --- Define network parameters ---
...

# --- Training loop ---
...

# --- Plot ---
...

# --- Save loss curve plots to q4_loss_curves.png ---
save_figure(fig, "q4_loss_curves")

# --- Prediction function ---
def predict(x):
    """Output the scalar prediction for a single training datapoint x"""
    ...

In [ ]:
grader.check("q4")

# Conclusion

Congrats!!! You've implemented PyTorch's autograd from scratch!

# Before you submit

Save the notebook, run every cell that produces a written answer or figure, and then run the check below. Missing files under `written/` print a warning but do not fail the cell.

In [ ]:
warn_missing_written_files()

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

## Submission

The cell below will generate a zip file for you to submit. It will also include all completed responses to manually graded questions saved under `written/` and other required files. **Please save before exporting!**


In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False, run_tests=True, files=['written'])